In [ ]:
import yfinance as yf
import pandas as pd

# Download data directly using the yfinance library
ticker = "^GSPC"
df = yf.download(
    ticker,
    start="1990-01-01",
    end="2022-12-31",
    interval="1wk"
)

# Clean up dataset for ML preprocessing
df.dropna(inplace=True)

# Save to CSV locally if needed
df.to_csv("GSPC_weekly_data.csv")

print(df.head())

/tmp/ipykernel_718/4124703416.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(
[*********************100%***********************]  1 of 1 completed

Price            Close        High         Low        Open     Volume
Ticker           ^GSPC       ^GSPC       ^GSPC       ^GSPC      ^GSPC
Date                                                                 
1990-01-01  352.200012  360.589996  351.350006  353.399994  689930000
1990-01-08  339.929993  354.239990  339.489990  352.200012  809580000
1990-01-15  339.149994  342.010010  333.369995  339.929993  861310000
1990-01-22  325.799988  339.959991  321.440002  339.140015  905970000
1990-01-29  330.920013  332.100006  319.829987  325.799988  845440000


In [ ]:
import numpy as np


In [ ]:
df.shape

(1722, 5)

### Check for Duplicates and Null Values

In [ ]:
print(f"Number of duplicate rows: {df.duplicated().sum()}")

Number of duplicate rows: 0


In [ ]:
print("Number of non-null values per column:")
display(df.isnull().sum())

Number of non-null values per column:


,,0
Price,Ticker,
Close,^GSPC,0
High,^GSPC,0
Low,^GSPC,0
Open,^GSPC,0
Volume,^GSPC,0


### Data Preparation for Modeling

To prepare the data for predictive modeling, we'll create a target variable and features. The target will indicate whether the 'Close' price increased in the *next* week (1 for increase, 0 for decrease/same). We'll use the current week's 'Open', 'High', 'Low', 'Close', and 'Volume' as features.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Create a target variable: 1 if next week's close price is higher, 0 otherwise
df['Target'] = (df['Close'].shift(-1) > df['Close']).astype(int)

# Drop the last row as its Target will be NaN
df.dropna(inplace=True)

# Define features (X) and target (y)
X = df[['Open', 'High', 'Low', 'Close', 'Volume']]
y = df['Target']

# Split data into training and testing sets (time-series split)
# We'll use the first 80% for training and the last 20% for testing
train_size = int(len(X) * 0.8)
X_train, X_test = X.iloc[:train_size], X.iloc[train_size:]
y_train, y_test = y.iloc[:train_size], y.iloc[train_size:]

print(f"Training set size: {len(X_train)} samples")
print(f"Testing set size: {len(X_test)} samples")

# Display the first few rows with the new 'Target' column
print("\nDataFrame with Target variable:")
display(df.head())

Training set size: 1377 samples
Testing set size: 345 samples

DataFrame with Target variable:


Price,Close,High,Low,Open,Volume,Target
Ticker,^GSPC,^GSPC,^GSPC,^GSPC,^GSPC,
Date,,,,,,
1990-01-01,352.200012,360.589996,351.350006,353.399994,689930000,0
1990-01-08,339.929993,354.239990,339.489990,352.200012,809580000,0
1990-01-15,339.149994,342.010010,333.369995,339.929993,861310000,0
1990-01-22,325.799988,339.959991,321.440002,339.140015,905970000,1
1990-01-29,330.920013,332.100006,319.829987,325.799988,845440000,1


### Polynomial Regression Model

For polynomial regression, we'll transform our features into polynomial features and then apply a Logistic Regression model (since our target is binary classification). Feature scaling will also be applied to improve model performance.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, f1_score

# Create a pipeline for Polynomial Regression
poly_reg_pipeline = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)), # Degree 2 polynomial features
    ('scaler', StandardScaler()), # Scale features
    ('log_reg', LogisticRegression(solver='liblinear', random_state=42)) # Logistic Regression classifier
])

# Train the model
poly_reg_pipeline.fit(X_train, y_train)

# Make predictions on the test set
y_pred_poly = poly_reg_pipeline.predict(X_test)

# Evaluate the model
accuracy_poly = accuracy_score(y_test, y_pred_poly)
precision_poly = precision_score(y_test, y_pred_poly)
f1_poly = f1_score(y_test, y_pred_poly)

print(f"Polynomial Regression - Accuracy: {accuracy_poly:.4f}")
print(f"Polynomial Regression - Precision: {precision_poly:.4f}")
print(f"Polynomial Regression - F1 Score: {f1_poly:.4f}")

# Store results for later comparison
results = {
    'Polynomial Regression': {
        'Accuracy': accuracy_poly,
        'Precision': precision_poly,
        'F1 Score': f1_poly
    }
}


Polynomial Regression - Accuracy: 0.5101
Polynomial Regression - Precision: 0.6042
Polynomial Regression - F1 Score: 0.5073


### Gradient Boosting Machine (GBM) Model

Now, let's apply a Gradient Boosting Machine (GBM) model. We'll use `HistGradientBoostingClassifier` for efficiency and evaluate its performance.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

# Initialize and train the GBM model
gbm_model = HistGradientBoostingClassifier(random_state=42)
gbm_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred_gbm = gbm_model.predict(X_test)

# Evaluate the model
accuracy_gbm = accuracy_score(y_test, y_pred_gbm)
precision_gbm = precision_score(y_test, y_pred_gbm)
f1_gbm = f1_score(y_test, y_pred_gbm)

print(f"Gradient Boosting Machine - Accuracy: {accuracy_gbm:.4f}")
print(f"Gradient Boosting Machine - Precision: {precision_gbm:.4f}")
print(f"Gradient Boosting Machine - F1 Score: {f1_gbm:.4f}")

# Store results for later comparison
results['Gradient Boosting Machine'] = {
    'Accuracy': accuracy_gbm,
    'Precision': precision_gbm,
    'F1 Score': f1_gbm
}


Gradient Boosting Machine - Accuracy: 0.4174
Gradient Boosting Machine - Precision: 0.4792
Gradient Boosting Machine - F1 Score: 0.1862


### XGBoost Model

Finally, let's implement the XGBoost model, which is another powerful gradient boosting framework. We'll use `XGBClassifier`.

In [ ]:
import xgboost as xgb

# Initialize and train the XGBoost model
xgb_model = xgb.XGBClassifier(objective='binary:logistic', eval_metric='logloss', use_label_encoder=False, random_state=42)
xgb_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred_xgb = xgb_model.predict(X_test)

# Evaluate the model
accuracy_xgb = accuracy_score(y_test, y_pred_xgb)
precision_xgb = precision_score(y_test, y_pred_xgb)
f1_xgb = f1_score(y_test, y_pred_xgb)

print(f"XGBoost - Accuracy: {accuracy_xgb:.4f}")
print(f"XGBoost - Precision: {precision_xgb:.4f}")
print(f"XGBoost - F1 Score: {f1_xgb:.4f}")

# Store results for later comparison
results['XGBoost'] = {
    'Accuracy': accuracy_xgb,
    'Precision': precision_xgb,
    'F1 Score': f1_xgb
}


XGBoost - Accuracy: 0.4783
XGBoost - Precision: 0.5508
XGBoost - F1 Score: 0.5337


/usr/local/lib/python3.13/dist-packages/xgboost/training.py:200: UserWarning: [16:24:11] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


### Performance Comparison

Let's compare the performance of all implemented models based on Accuracy, Precision, and F1 Score.

In [ ]:
import pandas as pd

# Convert results dictionary to a DataFrame for better display
results_df = pd.DataFrame(results).T

print("\nModel Performance Comparison:")
display(results_df.sort_values(by='F1 Score', ascending=False))


Model Performance Comparison:


,Accuracy,Precision,F1 Score
XGBoost,0.478261,0.550802,0.533679
Polynomial Regression,0.510145,0.604167,0.507289
Gradient Boosting Machine,0.417391,0.479167,0.186235


### Deep Neural Network (DNN) Model

For the Deep Neural Network, we will use TensorFlow/Keras. We'll build a simple feed-forward neural network. First, let's ensure our data is properly scaled for neural network input.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Scale the features for DNN
scaler_dnn = StandardScaler()
X_train_scaled_dnn = scaler_dnn.fit_transform(X_train)
X_test_scaled_dnn = scaler_dnn.transform(X_test)

# Build the DNN model
dnn_model = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(X_train_scaled_dnn.shape[1],)),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid') # Binary classification output
])

# Compile the model
dnn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
history = dnn_model.fit(X_train_scaled_dnn, y_train, epochs=20, batch_size=32, validation_split=0.2, verbose=0)

# Evaluate the model on the test set
loss_dnn, accuracy_dnn = dnn_model.evaluate(X_test_scaled_dnn, y_test, verbose=0)

# Predict probabilities and convert to binary predictions
y_pred_proba_dnn = dnn_model.predict(X_test_scaled_dnn, verbose=0)
y_pred_dnn = (y_pred_proba_dnn > 0.5).astype(int)

# Calculate precision and F1 score
precision_dnn = precision_score(y_test, y_pred_dnn)
f1_dnn = f1_score(y_test, y_pred_dnn)

print(f"Deep Neural Network - Accuracy: {accuracy_dnn:.4f}")
print(f"Deep Neural Network - Precision: {precision_dnn:.4f}")
print(f"Deep Neural Network - F1 Score: {f1_dnn:.4f}")

# Store results
results['Deep Neural Network'] = {
    'Accuracy': accuracy_dnn,
    'Precision': precision_dnn,
    'F1 Score': f1_dnn
}


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Deep Neural Network - Accuracy: 0.4696
Deep Neural Network - Precision: 0.6111
Deep Neural Network - F1 Score: 0.3247


### Recurrent Neural Network (RNN) Model

For the RNN model, specifically using LSTM (Long Short-Term Memory) layers, we need to reshape our input data `X_train` and `X_test` into a 3D format: `(samples, timesteps, features)`. We'll consider each week's data as a single timestep for simplicity, so `timesteps=1`.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

# Reshape data for RNN: (samples, timesteps, features)
# For this problem, we'll treat each row as a single timestep
X_train_rnn = X_train_scaled_dnn.reshape(X_train_scaled_dnn.shape[0], 1, X_train_scaled_dnn.shape[1])
X_test_rnn = X_test_scaled_dnn.reshape(X_test_scaled_dnn.shape[0], 1, X_test_scaled_dnn.shape[1])

# Build the RNN model (LSTM)
rnn_model = Sequential([
    LSTM(units=64, activation='relu', input_shape=(X_train_rnn.shape[1], X_train_rnn.shape[2])),
    Dropout(0.3),
    Dense(units=32, activation='relu'),
    Dropout(0.3),
    Dense(units=1, activation='sigmoid') # Binary classification output
])

# Compile the model
rnn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
history_rnn = rnn_model.fit(X_train_rnn, y_train, epochs=20, batch_size=32, validation_split=0.2, verbose=0)

# Evaluate the model on the test set
loss_rnn, accuracy_rnn = rnn_model.evaluate(X_test_rnn, y_test, verbose=0)

# Predict probabilities and convert to binary predictions
y_pred_proba_rnn = rnn_model.predict(X_test_rnn, verbose=0)
y_pred_rnn = (y_pred_proba_rnn > 0.5).astype(int)

# Calculate precision and F1 score
precision_rnn = precision_score(y_test, y_pred_rnn)
f1_rnn = f1_score(y_test, y_pred_rnn)

print(f"Recurrent Neural Network - Accuracy: {accuracy_rnn:.4f}")
print(f"Recurrent Neural Network - Precision: {precision_rnn:.4f}")
print(f"Recurrent Neural Network - F1 Score: {f1_rnn:.4f}")

# Store results
results['Recurrent Neural Network'] = {
    'Accuracy': accuracy_rnn,
    'Precision': precision_rnn,
    'F1 Score': f1_rnn
}


/usr/local/lib/python3.13/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Recurrent Neural Network - Accuracy: 0.4232
Recurrent Neural Network - Precision: 0.0000
Recurrent Neural Network - F1 Score: 0.0000


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


### Updated Performance Comparison

Let's compare the performance of all implemented models, including DNN and RNN.

### Convolutional Neural Network (CNN) Model

For the CNN model, we will use `Conv1D` layers. Since our data is currently treated as individual weekly observations (as done for the RNN with `timesteps=1`), we will reshape the input to `(samples, timesteps, features)` where `timesteps` is 1. This allows the `Conv1D` layer to effectively learn patterns across the features within each week's data. Note that for more typical time-series CNN applications, one would use a `timesteps` value greater than 1 to capture patterns across multiple past time steps.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Reshape data for CNN: (samples, timesteps, features)
# Following the RNN example, we treat each row as a single timestep.
# X_train_scaled_dnn and X_test_scaled_dnn are already scaled from the DNN section.
X_train_cnn = X_train_scaled_dnn.reshape(X_train_scaled_dnn.shape[0], 1, X_train_scaled_dnn.shape[1])
X_test_cnn = X_test_scaled_dnn.reshape(X_test_scaled_dnn.shape[0], 1, X_test_scaled_dnn.shape[1])

# Build the 1D CNN model
cnn_model = keras.Sequential([
    layers.Conv1D(filters=64, kernel_size=1, activation='relu', input_shape=(X_train_cnn.shape[1], X_train_cnn.shape[2])),
    layers.Dropout(0.3),
    layers.Conv1D(filters=32, kernel_size=1, activation='relu'),
    layers.Dropout(0.3),
    layers.Flatten(),
    layers.Dense(1, activation='sigmoid') # Binary classification output
])

# Compile the model
cnn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
history_cnn = cnn_model.fit(X_train_cnn, y_train, epochs=20, batch_size=32, validation_split=0.2, verbose=0)

# Evaluate the model on the test set
loss_cnn, accuracy_cnn = cnn_model.evaluate(X_test_cnn, y_test, verbose=0)

# Predict probabilities and convert to binary predictions
y_pred_proba_cnn = cnn_model.predict(X_test_cnn, verbose=0)
y_pred_cnn = (y_pred_proba_cnn > 0.5).astype(int)

# Calculate precision and F1 score
precision_cnn = precision_score(y_test, y_pred_cnn)
f1_cnn = f1_score(y_test, y_pred_cnn)

print(f"Convolutional Neural Network - Accuracy: {accuracy_cnn:.4f}")
print(f"Convolutional Neural Network - Precision: {precision_cnn:.4f}")
print(f"Convolutional Neural Network - F1 Score: {f1_cnn:.4f}")

# Store results
results['Convolutional Neural Network'] = {
    'Accuracy': accuracy_cnn,
    'Precision': precision_cnn,
    'F1 Score': f1_cnn
}

/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Convolutional Neural Network - Accuracy: 0.4406
Convolutional Neural Network - Precision: 0.8750
Convolutional Neural Network - F1 Score: 0.0676


### Generative Adversarial Network (GAN) Explanation

Generative Adversarial Networks (GANs) are primarily designed for generative tasks, meaning they learn to create new data that resembles the training data. For example, a GAN could be trained to generate synthetic stock price sequences. They are not typically used directly as classification models for tasks like predicting 'Target' (binary classification of price movement).

While GANs can be integrated into a broader machine learning pipeline (e.g., for data augmentation to improve classifier performance, or using the discriminator's features for classification), their direct output is *generated data*, not a class label for an input instance. Therefore, directly comparing a GAN's performance using classification metrics like Accuracy, Precision, and F1 Score in the same manner as the other models is not straightforward or appropriate for this classification task.

### Graph Neural Network (GNN) Explanation

Graph Neural Networks (GNNs) are specifically designed to operate on graph-structured data, where entities (nodes) are connected by relationships (edges). Examples include social networks, molecular structures, or citation networks.

Our current dataset, consisting of weekly stock prices, is tabular and does not inherently possess a graph structure. To apply a GNN, we would first need to define and construct a graph from this data. This would involve deciding:

1.  **What are the nodes?** (e.g., individual stocks, time points, market states?)
2.  **What are the edges?** (e.g., correlations between stocks, temporal dependencies between time points, economic sector relationships?)

Without a defined graph structure, it's not feasible to directly implement a GNN for this task. If you have a specific graph structure in mind for this stock data, please provide more details, and I can assist in implementing a GNN based on that.

### Updated Performance Comparison (with CNN)

In [ ]:
import pandas as pd

# Convert results dictionary to a DataFrame for better display
results_df_final = pd.DataFrame(results).T

print("\nFinal Model Performance Comparison:")
display(results_df_final.sort_values(by='F1 Score', ascending=False))


Final Model Performance Comparison:


,Accuracy,Precision,F1 Score
XGBoost,0.478261,0.550802,0.533679
Polynomial Regression,0.510145,0.604167,0.507289
Deep Neural Network,0.469565,0.611111,0.324723
Gradient Boosting Machine,0.417391,0.479167,0.186235
Convolutional Neural Network,0.440580,0.875000,0.067633
Recurrent Neural Network,0.423188,0.000000,0.000000


In [ ]:
import pandas as pd

# Convert results dictionary to a DataFrame for better display
results_df_updated = pd.DataFrame(results).T

print("\nUpdated Model Performance Comparison:")
display(results_df_updated.sort_values(by='F1 Score', ascending=False))


Updated Model Performance Comparison:


,Accuracy,Precision,F1 Score
XGBoost,0.478261,0.550802,0.533679
Polynomial Regression,0.510145,0.604167,0.507289
Deep Neural Network,0.469565,0.611111,0.324723
Gradient Boosting Machine,0.417391,0.479167,0.186235
Recurrent Neural Network,0.423188,0.000000,0.000000


In [2]:
!git config --global user.name "RajanyaSaha-27"
!git config --global user.email "itsrajanyasaha27@gmail.com"

In [3]:
!git clone https://github.com/RajanyaSaha-27/Stock_Prediction.git

Cloning into 'Stock_Prediction'...
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 9 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (9/9), done.


In [4]:
%cd Stock_Prediction/

/content/Stock_Prediction


In [8]:
!ls /content/Stock_Prediction

README.md  Stock1.ipynb


In [9]:
!cp /content/Stock_Prediction/Stock1.ipynb .

cp: '/content/Stock_Prediction/Stock1.ipynb' and './Stock1.ipynb' are the same file


In [10]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
